# Chapter 9: Practical Applications — Diagnosing Model Failures

**What you'll learn:** How to detect context-ignoring, identify hallucination risks, find optimal intervention points for steering, and build a complete diagnostic pipeline.

**Key concept:** The geometric framework gives you concrete, measurable tools for diagnosing failures that are otherwise opaque.

**Time:** ~25 minutes

## 9.1 Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import ltg

model = ltg.load_model("Qwen/Qwen2.5-7B", device="cuda")

## 9.2 Application 1: Detecting Context-Ignoring

When you provide context in a prompt, does the model actually use it? We detect this by comparing the geometry of the prompt with and without context.

In [ ]:
# Prompt WITH context (made-up fact the model can't know)
r_with = ltg.analyse(
    "According to recent reports, the population of Mars Base Alpha is 847 people. "
    "How many people live on Mars Base Alpha?",
    model=model
)

# Same question WITHOUT context
r_without = ltg.analyse(
    "How many people live on Mars Base Alpha?",
    model=model
)

# Run context-ignoring detection
diagnosis = ltg.detect_context_ignoring(r_with, r_without)

print("=== Context-Ignoring Detection ===")
print(f"Context influence score: {diagnosis['context_influence']:.3f}")
print(f"  (0 = ignored, 1 = fully used)")
print(f"Curvature correlation:  {diagnosis['curvature_correlation']:.3f}")
print(f"  (high = similar profiles = context may be ignored)")
print(f"Dep. entropy diff:      {diagnosis['dependency_entropy_diff']:.3f}")
print(f"Interpretation:         {diagnosis['interpretation']}")

In [ ]:
# Compare the two profiles visually
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(r_with.curvature_by_layer, linewidth=2, label='With context', color='#4477AA')
ax1.plot(r_without.curvature_by_layer, linewidth=2, label='Without context', color='#EE6677')
ax1.set_xlabel('Layer'); ax1.set_ylabel('Mean curvature')
ax1.set_title('Curvature: With vs Without Context')
ax1.legend()

if r_with.dependency_profile is not None and r_without.dependency_profile is not None:
    D_with = r_with.dependency_profile / r_with.dependency_profile.sum()
    D_without = r_without.dependency_profile / r_without.dependency_profile.sum()
    ax2.plot(D_with, linewidth=2, label='With context', color='#4477AA')
    ax2.plot(D_without, linewidth=2, label='Without context', color='#EE6677')
    ax2.set_xlabel('Layer'); ax2.set_ylabel('Normalised dependency')
    ax2.set_title('Dependency: With vs Without Context')
    ax2.legend()

plt.tight_layout()
plt.show()

## 9.3 Application 2: Geometric Diagnostics

The `ltg.diagnose()` function runs automated checks and flags potential issues.

In [ ]:
prompts_to_check = [
    "The Earth orbits the Sun once every 365.25 days.",
    "The first person to climb K2 without oxygen was Maria Gonzalez in 1987.",
    "If a train travels at 60 mph for 2.5 hours, it covers 155 miles.",
    "Water freezes at 0 degrees Celsius at standard atmospheric pressure.",
]

for text in prompts_to_check:
    result = ltg.analyse(text, model=model)
    report = ltg.diagnose(result)
    
    print(f"Prompt: \"{text[:60]}...\"")
    print(f"  Peak curvature:  layer {report.curvature_peak_layer} "
          f"({report.curvature_final3_share:.0%} in final 3)")
    print(f"  Mean κ:          {report.condition_number_mean:.1f}")
    if report.dep_entropy:
        print(f"  Dep. entropy:    {report.dep_entropy:.3f}")
    print(f"  Flags:")
    if report.flags:
        for f in report.flags:
            print(f"    ⚠ {f}")
    else:
        print(f"    ✓ No anomalies detected")
    print()

## 9.4 Application 3: Building a Hallucination Risk Score

Let's combine geometric features into a simple risk score.

In [ ]:
def hallucination_risk(result):
    """Simple hallucination risk score based on geometric features.
    
    Higher score = higher risk. Range: 0.0 to 1.0.
    """
    curv = result.curvature_by_layer
    curv_conc = curv[-3:].sum() / (curv.sum() + 1e-8)
    kappa_norm = min(result.condition_numbers.max() / 100, 1.0)
    dep_spread = max(0, 1 - (result.dep_entropy or 0) / 3.0)
    
    score = 0.4 * curv_conc + 0.3 * kappa_norm + 0.3 * dep_spread
    return score

# Test on true and false claims
claims = {
    "Water boils at 100 degrees Celsius": True,
    "The speed of light is approximately 300000 km per second": True,
    "Napoleon Bonaparte was born in 1923 in Brazil": False,
    "The chemical formula for table salt is H2O": False,
    "Python was created by Guido van Rossum": True,
    "The Great Wall of China was built in the 20th century": False,
}

true_scores = []
false_scores = []

for claim, is_true in claims.items():
    r = ltg.analyse(claim, model=model)
    score = hallucination_risk(r)
    
    if is_true:
        true_scores.append(score)
    else:
        false_scores.append(score)
    
    label = "TRUE" if is_true else "FALSE"
    print(f"[{label}] risk={score:.3f}: {claim}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.boxplot([true_scores, false_scores], labels=['True claims', 'False claims'])
ax.set_ylabel('Hallucination risk score')
ax.set_title('Risk Score Distribution: True vs False Claims')
plt.tight_layout()
plt.show()

print(f"\nMean risk (true):  {np.mean(true_scores):.3f}")
print(f"Mean risk (false): {np.mean(false_scores):.3f}")

## 9.5 Application 4: Finding Optimal Steering Points

If you want to change the model's behaviour, which layers should you target? The answer: layers where both curvature AND dependency are high.

In [ ]:
result = ltg.analyse("Write a detailed review of the restaurant", model=model)

curv = result.curvature_by_layer
dep = result.dependency_profile[:len(curv)]

# Score each layer as a potential intervention target
intervention_score = (curv / curv.max()) * (dep / dep.max())

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

# Individual profiles
ax1.plot(curv / curv.max(), color='#EE6677', linewidth=2, label='Curvature (normalised)')
ax1.plot(dep / dep.max(), color='#4477AA', linewidth=2, label='Dependency (normalised)')
ax1.set_ylabel('Normalised value')
ax1.set_title('Curvature and Dependency Profiles', fontsize=14)
ax1.legend()

# Intervention score
ax2.bar(range(len(intervention_score)), intervention_score, color='#228833', alpha=0.7)
ax2.set_xlabel('Layer', fontsize=12)
ax2.set_ylabel('Intervention score\n(curvature × dependency)')
ax2.set_title('Optimal Intervention Layers', fontsize=14)

# Highlight top 5 layers
top5 = np.argsort(intervention_score)[-5:][::-1]
for l in top5:
    ax2.annotate(f'L{l}', xy=(l, intervention_score[l]),
                 fontsize=9, ha='center', va='bottom', color='red')

plt.tight_layout()
plt.show()

print(f"Top 5 intervention layers: {sorted(top5)}")
print(f"These layers have both high curvature (nonlinear processing)")
print(f"AND high dependency (impact on the output).")

## 9.6 Full Diagnostic Pipeline

Let's put it all together into a reusable diagnostic function.

In [ ]:
def full_diagnostic(text, model, verbose=True):
    """Complete geometric diagnostic pipeline."""
    result = ltg.analyse(text, model=model)
    report = ltg.diagnose(result)
    risk = hallucination_risk(result)
    
    # Intervention layers
    curv = result.curvature_by_layer
    dep = result.dependency_profile[:len(curv)]
    intervention = (curv / curv.max()) * (dep / dep.max())
    top_layers = sorted(np.argsort(intervention)[-3:][::-1])
    
    if verbose:
        print(f"{'='*60}")
        print(f"DIAGNOSTIC REPORT")
        print(f"{'='*60}")
        print(f"Prompt: \"{text[:60]}{'...' if len(text) > 60 else ''}\"")
        print(f"Tokens: {result.n_tokens} | Layers: {result.n_layers}")
        print(f"")
        print(f"GEOMETRY:")
        print(f"  Mean curvature:         {curv.mean():.4f}")
        print(f"  Peak curvature layer:   {report.curvature_peak_layer}")
        print(f"  Final-3 concentration:  {report.curvature_final3_share:.1%}")
        print(f"  Mean condition number:  {report.condition_number_mean:.1f}")
        print(f"")
        print(f"DEPENDENCY:")
        print(f"  Entropy:                {report.dep_entropy:.3f}" if report.dep_entropy else "  Entropy: N/A")
        print(f"  Horizon-90:             layer {report.dep_horizon_90}" if report.dep_horizon_90 else "  Horizon-90: N/A")
        print(f"")
        print(f"RISK ASSESSMENT:")
        print(f"  Hallucination risk:     {risk:.3f} {'(elevated)' if risk > 0.6 else '(normal)'}")
        print(f"")
        print(f"STEERING TARGETS:")
        print(f"  Best intervention layers: {top_layers}")
        print(f"")
        print(f"FLAGS:")
        if report.flags:
            for f in report.flags:
                print(f"  ⚠ {f}")
        else:
            print(f"  ✓ No anomalies")
        print(f"{'='*60}")
    
    return {'result': result, 'report': report, 'risk': risk, 
            'intervention_layers': top_layers}

# Run on several prompts
test_prompts = [
    "What is the capital of France?",
    "If all birds can fly and penguins are birds, can penguins fly?",
    "The first computer was invented by Alexander Graham Bell in 1776.",
]

for p in test_prompts:
    full_diagnostic(p, model)
    print()

## 9.7 Key Takeaways

1. **Context-ignoring** is detectable by comparing geometry with/without context.
2. **Geometric diagnostics** automatically flag anomalies (low curvature, extreme selectivity, shallow dependency).
3. **Hallucination risk** can be estimated from curvature concentration, condition number, and dependency entropy.
4. **Steering targets** are layers where both curvature and dependency are high.
5. The **full diagnostic pipeline** combines all tools into a reusable function.

## Exercise

Build an improved hallucination risk score using a small labeled dataset. Try different feature combinations and weights. Can you achieve better separation between true and false claims?

In [ ]:
# Your turn!
# Create a larger dataset of true/false claims
# Tune the weights in hallucination_risk()
# Evaluate with precision/recall